# MPS Autoencoder — Colab Training Notebook
**Indus-2 Synchrotron · Magnet Power Supply Anomaly Detection**

Run this notebook on a free Colab GPU (Runtime → Change runtime type → T4 GPU).  
At the end, download `mps_autoencoder.weights.h5` and upload it in the Streamlit app → **Model Training → Load Saved Model**.

> **Why weights-only?** Saving the full model config embeds Keras version-specific fields (e.g. `quantization_config`) that cause `TypeError` when loaded by a different Keras version. Saving weights only is version-portable — the architecture is always rebuilt from code.

---
### Steps
1. Install / verify dependencies  
2. Generate synthetic dataset (or mount Drive to use real CSVs)  
3. Train autoencoder  
4. Evaluate + compute threshold  
5. Save weights and download

In [ ]:
# ── Cell 1: Dependencies ──────────────────────────────────────────────────────
!pip install -q tensorflow pandas numpy scikit-learn pyarrow
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Cell 2: Constants ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

NUM_SUPPLIES               = 117
ANOMALY_THRESHOLD_MULT     = 2.0
BEAM_PARTIAL_LOSS_THRESHOLD = 2.0
NUM_EVENTS                 = 10_000
NUM_TIMESTAMPS             = 50_000
SEED                       = 42
EPOCHS                     = 50      # more epochs than the local default
BATCH_SIZE                 = 512     # larger batch — GPU handles it fine
VAL_SPLIT                  = 0.2

In [ ]:
# ── Cell 3: Synthetic data generation ─────────────────────────────────────────
# If you have real CSVs, skip this cell and load them in Cell 4 instead.

rng = np.random.default_rng(SEED)

print('Generating event log…')
t_events = pd.date_range('2023-01-01', periods=NUM_EVENTS, freq='1min')
event_df = pd.DataFrame({
    'Time':        t_events,
    'Ramp end':    rng.choice(['Ramp done', ''], NUM_EVENTS, p=[0.35, 0.65]),
    'Kill signal': rng.choice(['Kill', ''],      NUM_EVENTS, p=[0.30, 0.70]),
    'Event type':  rng.choice(['Ramp done','Beam kill','Injection','Other'],
                               NUM_EVENTS, p=[0.25,0.25,0.25,0.25]),
})

print('Generating MPS signals…')
t_mps = pd.date_range('2023-01-01', periods=NUM_TIMESTAMPS, freq='1min')
beam_current = np.clip(
    100 + np.cumsum(rng.normal(0, 5, NUM_TIMESTAMPS)) * 0.01, 0, 120
)
fault_idx = rng.integers(5000, NUM_TIMESTAMPS - 5000, size=200)
for fi in fault_idx:
    beam_current[fi: fi + rng.integers(10, 60)] = rng.choice([beam_current[fi], 0.0])

mps_dict = {'Timestamp': t_mps, 'Beam Current': beam_current}
for i in range(1, NUM_SUPPLIES + 1):
    base  = rng.uniform(0.8, 1.2)
    noise = rng.normal(0, 0.005, NUM_TIMESTAMPS)
    mps_dict[f'sp{i}_vmeset']   = base + rng.normal(0, 0.002, NUM_TIMESTAMPS)
    mps_dict[f'sp{i}_readback'] = mps_dict[f'sp{i}_vmeset'] + noise

print('Injecting anomalies…')
anomaly_rows    = rng.choice(NUM_TIMESTAMPS, size=int(0.05*NUM_TIMESTAMPS), replace=False)
faulty_supplies = rng.integers(1, NUM_SUPPLIES+1, size=len(anomaly_rows))
for row, sup in zip(anomaly_rows, faulty_supplies):
    mps_dict[f'sp{sup}_readback'][row] += rng.uniform(0.05, 0.3)

mps_df = pd.DataFrame(mps_dict)
print(f'Done — {len(event_df):,} events, {len(mps_df):,} MPS timestamps')

In [ ]:
# ── Cell 4 (optional): Load real CSVs from Google Drive ───────────────────────
# Uncomment and run if you want to use real data instead of synthetic.

# from google.colab import drive
# drive.mount('/content/drive')

# event_df = pd.read_csv('/content/drive/MyDrive/indus2_events.csv', parse_dates=['Time'])
# mps_df   = pd.read_parquet('/content/drive/MyDrive/indus2_mps.parquet')
# print(f'Loaded {len(event_df):,} events, {len(mps_df):,} MPS timestamps')

In [ ]:
# ── Cell 5: Compute deviations ────────────────────────────────────────────────
print('Computing deviations…')
dev_dict = {'Timestamp': mps_df['Timestamp'], 'Beam Current': mps_df['Beam Current']}
for i in range(1, NUM_SUPPLIES + 1):
    dev_dict[f'sp{i}_dev'] = mps_df[f'sp{i}_readback'] - mps_df[f'sp{i}_vmeset']
dev_df = pd.DataFrame(dev_dict)

dev_cols = [c for c in dev_df.columns if c.endswith('_dev')]
X = dev_df[dev_cols].fillna(0).values.astype(np.float32)
print(f'Feature matrix: {X.shape}')  # (50000, 117)

In [ ]:
# ── Cell 6: Build autoencoder ─────────────────────────────────────────────────
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

def build_autoencoder(input_dim=NUM_SUPPLIES):
    inp = Input(shape=(input_dim,), name='input')
    x   = Dense(64,         activation='relu',   name='enc1')(inp)
    x   = Dense(32,         activation='relu',   name='enc2')(x)
    lat = Dense(16,         activation='relu',   name='latent')(x)
    x   = Dense(32,         activation='relu',   name='dec1')(lat)
    x   = Dense(64,         activation='relu',   name='dec2')(x)
    out = Dense(input_dim,  activation='linear', name='output')(x)
    model = Model(inp, out, name='MPS_Autoencoder')
    model.compile(optimizer='adam', loss='mae')
    return model

model = build_autoencoder()
model.summary()

In [ ]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

split   = int(len(X) * (1 - VAL_SPLIT))
X_train = X[:split]
X_val   = X[split:]
print(f'Train: {X_train.shape}  Val: {X_val.shape}')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

history = model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)
print('Training complete.')

In [ ]:
# ── Cell 8: Plot loss curves ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train MAE')
plt.plot(history.history['val_loss'], label='Val MAE',   linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('MAE'); plt.title('Autoencoder Training Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print(f'Best val MAE: {min(history.history["val_loss"]):.5f}')

In [ ]:
# ── Cell 9: Compute anomaly threshold ────────────────────────────────────────
preds_train = model.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
errors_train = np.mean(np.abs(X_train - preds_train), axis=1)

threshold = float(errors_train.mean() + ANOMALY_THRESHOLD_MULT * errors_train.std())
print(f'Anomaly threshold t_mps = {threshold:.5f}')

# Full-dataset inference
preds_all  = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
errors_all = np.mean(np.abs(X - preds_all), axis=1)
flags      = (errors_all > threshold).astype(int)
print(f'Anomalies: {flags.sum():,} / {len(flags):,}  ({100*flags.mean():.1f}%)')

In [ ]:
# ── Cell 10: Per-supply reconstruction error ──────────────────────────────────
per_supply_err = np.mean(np.abs(X - preds_all), axis=0)
top10 = np.argsort(per_supply_err)[::-1][:10]
print('Top-10 highest-error supplies:')
for rank, idx in enumerate(top10, 1):
    print(f'  {rank}. sp{idx+1}  mean |error| = {per_supply_err[idx]:.5f}',
          '  ← ANOMALOUS' if per_supply_err[idx] > threshold else '')

In [ ]:
# ── Cell 11: Save weights only ────────────────────────────────────────────────
# Weights-only format is portable across Keras versions.
# The Streamlit app rebuilds the architecture from code and loads these weights.
WEIGHTS_PATH = 'mps_autoencoder.weights.h5'
model.save_weights(WEIGHTS_PATH)
print(f'Saved → {WEIGHTS_PATH}')

import json
with open('mps_threshold.json', 'w') as f:
    json.dump({
        'threshold':      threshold,
        'epochs_trained': len(history.history['loss']),
        'final_val_mae':  min(history.history['val_loss']),
    }, f, indent=2)
print('Saved → mps_threshold.json')

In [ ]:
# ── Cell 12: Download from Colab ──────────────────────────────────────────────
from google.colab import files
files.download('mps_autoencoder.weights.h5')   # upload this in Streamlit → Load Saved Model
files.download('mps_threshold.json')           # reference value for t_mps
files.download('training_loss.png')

## After downloading

1. Open the Streamlit app
2. Go to **🧠 Model Training → Load Saved Model**
3. Upload `mps_autoencoder.weights.h5`
4. The app rebuilds the architecture and loads the weights — no config deserialization, no version mismatch

The threshold is recomputed from the current dataset on load.  
Check `mps_threshold.json` to compare the Colab-computed value.